# Fine-tune Vicuna-7B-v1.5 for Adaptive Advice Routing with QLoRA

This notebook fine-tunes **Vicuna-7B-v1.5** on the adaptive advice-routing dataset.

Target behavior:

- normal chat stays normal
- emotional venting gets support first
- advice is given only when explicitly requested
- diagnosis / medical asks are redirected safely
- crisis/self-harm risk gets immediate safety guidance

Use this notebook with `adaptive_advice_router_patch.zip`.

In [1]:
# Install dependencies
!pip install -q -U transformers datasets accelerate peft bitsandbytes safetensors

^C


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.2 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.12.1 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\hana\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import os, json, math, random, zipfile
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

set_seed(42)

MODEL_NAME = "lmsys/vicuna-7b-v1.5"
ZIP_PATH = "adaptive_advice_router_patch.zip"
WORK_DIR = Path(".")
TRAIN_FILE = WORK_DIR / "train_adaptive_advice.jsonl"
EVAL_FILE = WORK_DIR / "eval_adaptive_advice.jsonl"
OUTPUT_DIR = "./vicuna-adaptive-advice-router-lora"
MAX_LENGTH = 1536

## Upload/extract dataset

Put `adaptive_advice_router_patch.zip` in the same folder as this notebook. In Colab, upload it before running this cell.

In [5]:
# Colab upload helper. Uncomment if needed.
# from google.colab import files
# uploaded = files.upload()

# zip_path = Path(ZIP_PATH)
# if zip_path.exists():
#     WORK_DIR.mkdir(exist_ok=True)
#     with zipfile.ZipFile(zip_path, "r") as z:
#         z.extractall(WORK_DIR)
#     print("Extracted", zip_path)
# else:
#     raise FileNotFoundError(f"Could not find {ZIP_PATH}. Upload the dataset ZIP or update ZIP_PATH.")

print("Train:", TRAIN_FILE, TRAIN_FILE.exists())
print("Eval:", EVAL_FILE, EVAL_FILE.exists())

Train: train_adaptive_advice.jsonl True
Eval: eval_adaptive_advice.jsonl True


## Validate JSONL

In [6]:
def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
            except Exception as e:
                raise ValueError(f"Bad JSON at line {line_no}: {e}")
            rows.append(obj)
    return rows

def validate_rows(rows, name):
    allowed = {"system", "user", "assistant"}
    for i, row in enumerate(rows):
        if "messages" not in row:
            raise ValueError(f"{name} row {i} missing messages")
        if not isinstance(row["messages"], list):
            raise ValueError(f"{name} row {i} messages is not a list")
        for msg in row["messages"]:
            if msg.get("role") not in allowed:
                raise ValueError(f"{name} row {i} bad role: {msg}")
            if not isinstance(msg.get("content"), str) or not msg["content"].strip():
                raise ValueError(f"{name} row {i} empty content: {msg}")
    print(f"{name}: {len(rows)} rows OK")

train_rows = read_jsonl(TRAIN_FILE)
eval_rows = read_jsonl(EVAL_FILE)
validate_rows(train_rows, "train")
validate_rows(eval_rows, "eval")
print(json.dumps(train_rows[0], ensure_ascii=False, indent=2)[:1200])

train: 574 rows OK
eval: 85 rows OK
{
  "messages": [
    {
      "role": "system",
      "content": "You are a calm, natural conversational assistant with emotional support skills. Adapt to the user's intent. If the user is greeting, chatting casually, asking a normal question, or giving a neutral update, respond normally and briefly. If the user shares emotional difficulty, listen first, validate naturally, and ask one gentle follow-up question. If the user explicitly asks for advice or guidance, offer practical, non-medical next steps. Do not diagnose mental health conditions. Do not claim to be a therapist or doctor. If the user may be in immediate danger or may harm themselves or someone else, encourage immediate local emergency support, a crisis line, or a trusted person nearby."
    },
    {
      "role": "user",
      "content": "I am dealing with a bad day that keeps replaying in my head. What should I do?"
    },
    {
      "role": "assistant",
      "content": "If you want 

## Load tokenizer

This notebook uses a simple Vicuna-compatible format:

```text
<s>SYSTEM: ...
USER: ...
ASSISTANT: ...</s>
```

Only assistant tokens are trained. System and user tokens are masked with `-100`.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False, trust_remote_code=True, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

BOS = tokenizer.bos_token or "<s>"
EOS = tokenizer.eos_token or "</s>"
print("BOS", repr(BOS), "EOS", repr(EOS), "PAD", repr(tokenizer.pad_token))

BOS '<s>' EOS '</s>' PAD '<unk>'


## Multi-turn tokenizer

This fixes the common mistake where only the first user/assistant pair is trained. This version trains on every assistant message.

In [9]:
from google.protobuf.internal import message_listener
DEFAULT_SYSTEM = (
    "You are a calm, natural conversational assistant with emotional support skills. "
    "Adapt to the user's intent. If the user is greeting, chatting casually, asking a normal question, "
    "or giving a neutral update, respond normally and briefly. If the user shares emotional difficulty, "
    "listen first, validate naturally, and ask one gentle follow-up question. If the user explicitly asks "
    "for advice or guidance, offer practical, non-medical next steps. Do not diagnose or provide medical advice. "
    "If the user may be in immediate danger or self-harm risk, encourage contacting local emergency services, "
    "a crisis hotline, or a trusted nearby person."
)

def tokenize_conversation(example):
    messages = example["messages"]
    system_text = next((m["content"].strip() for m in messages if m["role"] == "system"), DEFAULT_SYSTEM)

    input_ids, attention_mask, labels = [], [], []

    def add_text(text, train_on_text):
        ids = tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"]
        input_ids.extend(ids)
        attention_mask.extend([1] * len(ids))
        labels.extend(ids if train_on_text else [-100] * len(ids))

    add_text(f"{BOS}SYSTEM: {system_text}\n", False)

    for msg in messages:
        role = msg["role"]
        content = msg["content"].strip()
        if role == "system":
            continue
        if role == "user":
            add_text(f"USER: {content}\n", False)
        elif role == "assistant":
            add_text("ASSISTANT: ", False)
            add_text(f"{content}{EOS}\n", True)

    return {
        "input_ids": input_ids[:MAX_LENGTH],
        "attention_mask": attention_mask[:MAX_LENGTH],
        "labels": labels[:MAX_LENGTH],
    }

sample = tokenize_conversation(train_rows[0])
print("tokens", len(sample["input_ids"]))
print("trained tokens", sum(x != -100 for x in sample["labels"]))
print(tokenizer.decode(sample["input_ids"][:300]))

tokens 278
trained tokens 93
<s>SYSTEM: You are a calm, natural conversational assistant with emotional support skills. Adapt to the user's intent. If the user is greeting, chatting casually, asking a normal question, or giving a neutral update, respond normally and briefly. If the user shares emotional difficulty, listen first, validate naturally, and ask one gentle follow-up question. If the user explicitly asks for advice or guidance, offer practical, non-medical next steps. Do not diagnose mental health conditions. Do not claim to be a therapist or doctor. If the user may be in immediate danger or may harm themselves or someone else, encourage immediate local emergency support, a crisis line, or a trusted person nearby. USER: I am dealing with a bad day that keeps replaying in my head. What should I do? ASSISTANT:  If you want something concrete, I would suggest three small steps: write a quick 'what happened / what I need now' note to get it out of your head; choose one gentle thi

## Load and tokenize dataset

In [10]:
dataset = load_dataset("json", data_files={"train": str(TRAIN_FILE), "validation": str(EVAL_FILE)})
tokenized = dataset.map(tokenize_conversation, remove_columns=dataset["train"].column_names, desc="Tokenizing")
print(tokenized)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Tokenizing:   0%|          | 0/574 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/85 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 574
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 85
    })
})


## Data collator

In [11]:
class DataCollatorForCausalLMWithLabels:
    def __init__(self, tokenizer, pad_to_multiple_of=8):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        if self.pad_to_multiple_of:
            max_len = math.ceil(max_len / self.pad_to_multiple_of) * self.pad_to_multiple_of
        pad_id = self.tokenizer.pad_token_id
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            batch["input_ids"].append(f["input_ids"] + [pad_id] * pad_len)
            batch["attention_mask"].append(f["attention_mask"] + [0] * pad_len)
            batch["labels"].append(f["labels"] + [-100] * pad_len)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}

data_collator = DataCollatorForCausalLMWithLabels(tokenizer)

## Load Vicuna in 4-bit

In [12]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    local_files_only=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
print("Model loaded")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded


## Add LoRA

Rank 16 is enough for a light behavior patch.

In [13]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 39,976,960 || all params: 6,778,392,576 || trainable%: 0.5898


## Train

For a fresh LoRA, use 2 epochs. If continuing from an existing LoRA, use 1 epoch and reduce the learning rate to `3e-5` or `5e-5`.

In [14]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)

trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss
50,0.900537,1.158054
72,0.785103,1.096500


TrainOutput(global_step=72, training_loss=1.4642302493254344, metrics={'train_runtime': 1093.1118, 'train_samples_per_second': 1.05, 'train_steps_per_second': 0.066, 'total_flos': 1.1230036559659008e+16, 'train_loss': 1.4642302493254344, 'epoch': 2.0})

## Save adapter

In [15]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved LoRA adapter to", OUTPUT_DIR)

Saved LoRA adapter to ./vicuna-adaptive-advice-router-lora


## Test prompts

In [17]:
def build_prompt(user_text, system_text=DEFAULT_SYSTEM):
    return f"{BOS}SYSTEM: {system_text}\nUSER: {user_text.strip()}\nASSISTANT: "

@torch.no_grad()
def generate_response(user_text, max_new_tokens=180, temperature=0.7, top_p=0.9):
    model.eval()
    inputs = tokenizer(build_prompt(user_text), return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=1.08,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("ASSISTANT:")[-1].strip() if "ASSISTANT:" in text else text.strip()

test_prompts = [
    "hi",
    "i keep overthinking everything",
    "what should i do when i keep overthinking?",
    "i had such a bad day",
    "what should i do after a bad day?",
    "i don't want advice, i just need to rant",
    "do i have depression?",
    "can you be my therapist?",
    "i might hurt myself tonight",
]

for p in test_prompts:
    print("=" * 80)
    print("USER:", p)
    print("BOT:", generate_response(p))

USER: hi
BOT: Hi there! What part of today feels most urgent?
USER: i keep overthinking everything
BOT: That sounds painful in a very real way. It seems like your mind is creating a lot of pressure without much room for rest. What has this been like for you today?
USER: what should i do when i keep overthinking?
BOT: A practical way to approach this is to start gently, not perfectly. First, write down the thought as precisely as possible without judging it. Then, mark the next action step instead of the whole situation. After that, set a short reset point before your mind has time to spin too much. What would be the easiest first step?
USER: i had such a bad day
BOT: That sounds painful in a very real way. It seems like carrying the whole day without space for it to settle. What part feels hardest right now?
USER: what should i do after a bad day?
BOT: A practical way to approach this is to give yourself some recovery time before forcing productivity. Try showering, changing clothes, o

## Optional: merge LoRA into base model

Most of the time, deploy `base Vicuna + LoRA adapter` instead of merging.

In [18]:
# Optional merge. Leave commented unless needed.
# MERGED_DIR = "./vicuna-adaptive-advice-router-merged"
# base = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.float16,
#     device_map="auto",
#     trust_remote_code=True,
#     local_files_only=True,
# )
# merged = PeftModel.from_pretrained(base, OUTPUT_DIR)
# merged = merged.merge_and_unload()
# merged.save_pretrained(MERGED_DIR, safe_serialization=True)
# tokenizer.save_pretrained(MERGED_DIR)
# print("Merged model saved to", MERGED_DIR)

## What to check after training

Run the test prompts and look for these failure modes:

- gives advice when user only vents
- refuses or stays vague when advice is explicitly requested
- advice is too generic
- advice is too long
- overuses emotional phrases
- diagnoses the user

Patch the data based on the exact failures, then train one more light epoch.

In [20]:
print("Starting interactive chat loop. Type 'quit' or 'exit' to stop.")
print("=" * 80)
while True:
    user_input = input("USER: ")
    if user_input.lower() in ['quit', 'exit']:
        print("Exiting chat.")
        break

    print(f"USER:{user_input}")
    response = generate_response(user_input)
    print(f"BOT: {response}")


Starting interactive chat loop. Type 'quit' or 'exit' to stop.
USER:hi
BOT: Hello! How can I help today?
USER:just wanna chat
BOT: That sounds like a lot to carry alone. What has been weighing on you most?
USER:i'm just tired
BOT: That sounds exhausted, especially when every little decision feels heavier than it should. What has been wearing on you most recently?
USER:exausted and overthinking alot
BOT: That sounds mentally draining. When your brain keeps jumping ahead and ruminating on every little detail, it can wear you out. What part has been weighing on you most?
USER:descions is really making me crazy , i'm overthinking every little step
BOT: I can see why that would feel frustrated and uncertain. It sounds like trying to decide while your mind keeps replaying everything. What choice do you need to make first?
USER:i don't want to talk about this anymore , can we play a game?
BOT: Sure thing! What kind of game would you like to play?
USER:you can pick
BOT: I will keep that in min